# ConformerFlow — Session 2 Training (Epochs 51-100)

**Before running:**
1. `Settings → Accelerator → GPU T4 x2`
2. `Settings → Turn on Internet → ON`
3. Add datasets as input:
   - `perinbamkumar/conformerflow-ckpt-session1` (checkpoint)
   - `perinbamkumar/conformerflow-data` (parsed data)

**Run cells top to bottom.**

In [ ]:
# Cell 1 — Clone repo and install
!git clone https://github.com/kumar23kan/conformerflow.git /kaggle/working/conformerflow
!pip install -q biopython gemmi requests tqdm numpy einops scipy

In [ ]:
# Cell 2 — Set paths
import os, sys
os.chdir('/kaggle/working/conformerflow/files')
sys.path.insert(0, '/kaggle/working/conformerflow/files')
print('Working directory:', os.getcwd())

In [ ]:
# Cell 3 — Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 4 — Load data from Kaggle dataset input
import shutil
from pathlib import Path

# Create directories
for d in ['pdb_data/parsed_nmr', 'pdb_data/splits', 'checkpoints']:
    Path(d).mkdir(parents=True, exist_ok=True)

# Copy parsed data from input dataset
data_input = '/kaggle/input/conformerflow-data'
if os.path.exists(data_input):
    shutil.copytree(f'{data_input}/parsed_nmr', 'pdb_data/parsed_nmr', dirs_exist_ok=True)
    shutil.copytree(f'{data_input}/splits', 'pdb_data/splits', dirs_exist_ok=True)
    print('Data loaded from dataset')
else:
    print('ERROR: conformerflow-data dataset not attached. Add it in Settings → Add Data.')

# Check data
import json
for split in ['train', 'val', 'test']:
    path = f'pdb_data/splits/{split}.json'
    if os.path.exists(path):
        data = json.load(open(path))
        print(f'{split}: {len(data)} entries')

In [ ]:
# Cell 5 — Load checkpoint from previous session
ckpt_input = '/kaggle/input/conformerflow-ckpt-session1/checkpoints_saved/ckpt_best.pt'

if os.path.exists(ckpt_input):
    shutil.copy(ckpt_input, 'checkpoints/ckpt_resume.pt')
    print(f'Checkpoint loaded: {ckpt_input}')
else:
    print('ERROR: Checkpoint not found. Add conformerflow-ckpt-session1 dataset in Settings → Add Data.')

In [ ]:
# Cell 6 — Train epochs 51-100 (resume from checkpoint)
!python3 train.py \
    --config configs/base_config.yaml \
    --batch_size 2 \
    --max_epochs 100 \
    --max_residues 300 \
    --resume_from checkpoints/ckpt_resume.pt

In [ ]:
# Cell 7 — Save checkpoints to output
import shutil, os

src = '/kaggle/working/conformerflow/files/checkpoints'
dst = '/kaggle/working/checkpoints_saved'

if os.path.exists(src) and os.listdir(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print('Checkpoints saved:')
    for f in os.listdir(dst):
        size = os.path.getsize(os.path.join(dst, f)) / 1e6
        print(f'  {f}  ({size:.1f} MB)')
else:
    print('No checkpoints found.')